In [10]:
from PIL import Image  # Импорт библиотеки для работы с изображениями
from transformers import CLIPProcessor, CLIPModel  # Импорт модели и процессора CLIP
import pandas as pd  # Импорт библиотеки для работы с данными
import matplotlib.pyplot as plt  # Импорт библиотеки для построения графиков
from sklearn.metrics import confusion_matrix  # Импорт функции для построения матрицы путаницы
import torch  # Импорт библиотеки для работы с тензорами и моделями
import numpy as np  # Импорт библиотеки для работы с массивами
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Модель и процессор
# Пути для сохранения модели и процессора
model_save_path = "../models/saved_clip_model"
processor_save_path = "../models/saved_clip_processor"

# 1. Загрузка модели и процессора (с проверкой наличия сохранённой версии)
model_name = "openai/clip-vit-large-patch14-336"

if os.path.exists(model_save_path) and os.path.exists(processor_save_path):
    print("Загружаем сохранённую модель и процессор с диска...")
    model = CLIPModel.from_pretrained(model_save_path).to(device)
    processor = CLIPProcessor.from_pretrained(processor_save_path)
else:
    print("Загружаем модель и процессор из Hugging Face, затем сохраняем на диск...")
    # Загружаем оригинальную модель
    model = CLIPModel.from_pretrained(model_name).to(device)
    processor = CLIPProcessor.from_pretrained(model_name)
    
    # Сохраняем на диск для будущего использования
    model.save_pretrained(model_save_path)
    processor.save_pretrained(processor_save_path)
    print(f"Модель сохранена в: {model_save_path}")
    print(f"Процессор сохранён в: {processor_save_path}")
# 2. Названия классов

id2label = {
    0: "multi-purpose room",
    1: "kitchen / dining room",
    2: "kitchen-living room",
    3: "living room",
    4: "bedroom",
    5: "study / home office",
    6: "children's room",
    7: "bathroom",
    8: "toilet",
    9: "combined bathroom and toilet",
    10: "hallway / entrance hall",
    11: "walk‑in closet / storage room / laundry room",
    12: "balcony / loggia",
    13: "view from the window / from the balcony",
    14: "outside the house / yard",
    15: "entrance hall / staircase landing",
    16: "other",
    17: "interior items / household appliances",
    18: "room without furniture",
    19: "unknown additional class"
}

# 3. Текстовые промпты
text_prompts = [
    f"photo: {label}"
    for _, label in sorted(id2label.items())
]

# 4. Загрузка картинки
image_path = "../data/room_type/train_images/train_images/11005325159.jpg"
image = Image.open(image_path).convert("RGB")

# 5. Подготовка входов
inputs = processor(
    text=text_prompts,
    images=image,
    return_tensors="pt",
    padding=True
).to(device)

# 6. Инференс
with torch.no_grad():
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=1)

pred_idx = probs.argmax(dim=1).item()
pred_label = id2label[pred_idx]
pred_prob = probs[0, pred_idx].item()

print("pred_idx:", pred_idx)
print("pred_label:", pred_label)
print("pred_prob:", round(pred_prob, 4))

Загружаем модель и процессор из Hugging Face, затем сохраняем на диск...


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.71s/it]

Модель сохранена в: ../models/saved_clip_model
Процессор сохранён в: ../models/saved_clip_processor
pred_idx: 2
pred_label: kitchen-living room
pred_prob: 0.4746
